# Padel AI Coach – train the models (Google Colab GPU)
Trains everything the local app uses for **raw, unlabelled match videos**:

| Model | File for `c:\Padel\models\` | Data |
|---|---|---|
| Ball detector (YOLO11s, 1280 px) | `padel_ball.onnx` | PadelTracker100 ball boxes (both finals) |
| Player 17-keypoint pose (YOLO11s-pose fine-tune) | `padel_pose.onnx` | PadelTracker100 poses |
| Temporal shot classifier (1D-CNN) | `padel_shot_classifier.onnx` + `.json` | PadelTracker100 shot labels |

**Before you start**
1. On the laptop: `python -m padelcoach export-training` → creates `colab_bundle/padel_bundle.zip`.
2. In Google Drive create `MyDrive/padel/` and upload `padel_bundle.zip` plus a folder `videos/` with `2022_BCN_FinalF_1.mp4` and `2022_BCN_FinalM_1.mp4`.
3. Runtime → Change runtime type → **T4 GPU**. Expect several hours in total (ball model is the longest). Checkpoints are saved to Drive every epoch:
   if Colab disconnects, reconnect, run the setup cells (GPU check, mount, install, datasets) again and re-run the training cell – it resumes.

Licences: PadelTracker100 is CC-BY-4.0. Ultralytics YOLO is AGPL-3.0 (fine for personal/internal use; a commercial closed product needs an Ultralytics licence or another detector).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/padel'
!mkdir -p /content/padel $DRIVE/models $DRIVE/runs
!unzip -oq $DRIVE/padel_bundle.zip -d /content/padel
%cd /content/padel
!ls

In [ ]:
!pip -q install ultralytics onnx onnxruntime onnxslim

## 1. Build the YOLO datasets (frames extracted from the videos, temporal train/val split)

In [ ]:
!python colab_scripts/prepare_yolo_datasets.py --videos $DRIVE/videos --labels labels --out /content/datasets

## 2. Ball detector
Small, fast and motion-blurred balls → high input resolution (1280) and little geometric augmentation.
*Optional:* more ball data (e.g. Roboflow Universe "padel-ball-detection", CC BY 4.0, needs a free API key) can be merged into `/content/datasets/ball` before training.

In [ ]:
import os
from ultralytics import YOLO
# Resumes automatically if Colab disconnected: just run this cell again.
last_ball = f'{DRIVE}/runs/ball/weights/last.pt'
if os.path.exists(last_ball):
    YOLO(last_ball).train(resume=True)
else:
    YOLO('yolo11s.pt').train(data='/content/datasets/ball.yaml', imgsz=1280, epochs=30, batch=8, patience=8,
                             mosaic=0.5, degrees=0, scale=0.3, fliplr=0.5, hsv_h=0.01, close_mosaic=5,
                             save_period=1, project=f'{DRIVE}/runs', name='ball', exist_ok=True)
best_ball = f'{DRIVE}/runs/ball/weights/best.pt'
print('ball mAP50:', YOLO(best_ball).val(data='/content/datasets/ball.yaml', imgsz=1280).box.map50)

In [ ]:
p = YOLO(best_ball).export(format='onnx', imgsz=1280, opset=17, simplify=True)
!cp {p} $DRIVE/models/padel_ball.onnx

## 3. Player pose (17 COCO keypoints) – fine-tune so far-court players are found reliably

In [ ]:
last_pose = f'{DRIVE}/runs/pose/weights/last.pt'
if os.path.exists(last_pose):
    YOLO(last_pose).train(resume=True)
else:
    YOLO('yolo11s-pose.pt').train(data='/content/datasets/pose.yaml', imgsz=1280, epochs=25, batch=8, patience=6,
                                  mosaic=0.5, degrees=0, fliplr=0.5, save_period=1,
                                  project=f'{DRIVE}/runs', name='pose', exist_ok=True)
best_pose = f'{DRIVE}/runs/pose/weights/best.pt'
p = YOLO(best_pose).export(format='onnx', imgsz=1280, opset=17, simplify=True)
!cp {p} $DRIVE/models/padel_pose.onnx

## 4. Shot classifier
Prints the **cross-match** accuracy (train on one final, test on the other). The local rule-based classifier scores ~80 % on the women's final;
install the ONNX model only if the cross-match accuracy here is higher (the app blends both when the model is confident).

In [ ]:
!python colab_scripts/train_shot_model.py --data shot_sequences.npz --out /content/models --epochs 120
!cp /content/models/padel_shot_classifier.onnx /content/models/padel_shot_classifier.json $DRIVE/models/

## 5. Install on the laptop
Download from `MyDrive/padel/models/` and put the files in **`c:\Padel\models\`**:
`padel_ball.onnx`, `padel_pose.onnx`, `padel_shot_classifier.onnx`, `padel_shot_classifier.json`.
Restart `python -m padelcoach serve`; the Matches page shows the models as installed. Keep `runs/*/weights/best.pt` for `padel_gpu_inference.ipynb`.

In [ ]:
!cd $DRIVE/models && zip -q -r /content/padel_models.zip . && ls -la $DRIVE/models
from google.colab import files
files.download('/content/padel_models.zip')